In [8]:
import pandas as pd
import os
print(os.getcwd())

# Carga de archivos
productos = pd.read_csv('productos_.csv')
clientes = pd.read_csv('clientes.csv')
ventas = pd.read_csv('ventas.csv')

c:\Users\ASUS\Downloads\dataset_examen


Pregunta 1 Líneas, tickets y unidades. ¿Cuántas líneas de venta, cuántos tickets distintos y cuántas unidades vendidas hay en el archivo? Explique qué representa cada conteo.

In [9]:
num_lineas = len(ventas)
num_tickets = ventas['id_ticket'].nunique()
total_unidades = ventas['cantidad'].sum()

print("=== Pregunta 1: Líneas, tickets y unidades ===")
print(f"Líneas de venta: {num_lineas}")
print(f"Tickets distintos: {num_tickets}")
print(f"Unidades vendidas: {total_unidades}")

=== Pregunta 1: Líneas, tickets y unidades ===
Líneas de venta: 2100
Tickets distintos: 300
Unidades vendidas: 3880


- Líneas de venta (2,100):** representan cada registro individual de un producto dentro de un ticket en `ventas.csv`. Si un ticket incluye 3 productos distintos, ocupa 3 líneas.
- Tickets distintos (300):** representan las compras o transacciones únicas realizadas en el punto de venta (identificadas por `id_ticket` sin repetir).
- Unidades vendidas (3,880):** representan la cantidad total de ítems físicos despachados al sumar la columna `cantidad` de todas las líneas de venta.

Pregunta 3 Métodos de pago. ¿Cuántos tickets se pagaron con cada método y cuántos tienen ese dato sin registrar?

In [10]:
ticket_pago = ventas[['id_ticket', 'metodo_pago']].drop_duplicates()
conteo_pagos = ticket_pago['metodo_pago'].fillna('Sin registrar').value_counts()
print(conteo_pagos)

metodo_pago
Efectivo         144
Tarjeta           84
Transferencia     48
Sin registrar     24
Name: count, dtype: int64



Conteo de tickets únicos por método de pago, incluyendo los registros sin método asignado (24 tickets).

Pregunta 5 Unidades por sucursal y categoría. ¿Cuántas unidades vendió cada sucursal de cada categoría? Presente una tabla cruzada con totales por fila y por columna. Compruebe que el total general coincide con la suma de cantidad en ventas.csv.

In [11]:
ventas_prod = ventas.merge(productos, on='id_producto', how='left')

tabla_cruzada = pd.pivot_table(
    ventas_prod,
    values='cantidad',
    index='sucursal',
    columns='categoria',
    aggfunc='sum',
    margins=True,
    margins_name='Total'
)
print(tabla_cruzada)

total_tabla = int(tabla_cruzada.loc['Total', 'Total'])
total_ventas = int(ventas['cantidad'].sum())
print(f"\nComprobación -> Total tabla: {total_tabla} | Suma cantidad en ventas.csv: {total_ventas} | Coincide: {total_tabla == total_ventas}")

categoria  Accesorios  Archivo  Escritura  Papel y cuadernos  Total
sucursal                                                           
Central           453      481        437                524   1895
Norte             256      264        306                259   1085
Sur               191      223        238                248    900
Total             900      968        981               1031   3880

Comprobación -> Total tabla: 3880 | Suma cantidad en ventas.csv: 3880 | Coincide: True



Tabla cruzada de unidades vendidas por sucursal y categoría con totales por fila y columna, comprobando que el total general (3,880) coincide con la suma de `cantidad` en `ventas.csv`.

Pregunta 6 Importe de ventas por sucursal. ¿Cuál fue el importe total de ventas de cada sucursal? Calcule primero el precio total de cada línea como cantidad × precio_unitario.

In [12]:
ventas['importe'] = ventas['cantidad'] * ventas['precio_unitario']
importe_sucursal = ventas.groupby('sucursal')['importe'].sum().reset_index()

print("=== Importe de ventas por sucursal ===")
for _, fila in importe_sucursal.iterrows():
    print(f"{fila['sucursal']}: ${fila['importe']:.2f}")

=== Importe de ventas por sucursal ===
Central: $3025.04
Norte: $1635.35
Sur: $1443.24



Importe total de ventas por sucursal calculado a partir de `cantidad * precio_unitario`, expresado en dólares con dos decimales.

Pregunta 9 Producto más vendido por sucursal. ¿Qué producto acumuló más unidades vendidas en cada sucursal? Indique su código, nombre y cantidad vendida. Si hay un empate, incluya todos los productos empatados.

In [13]:
vp_suc = ventas_prod.groupby(['sucursal', 'id_producto', 'producto'], as_index=False)['cantidad'].sum()
max_sucursal = vp_suc.groupby('sucursal')['cantidad'].transform('max')
producto_mas_vendido = vp_suc[vp_suc['cantidad'] == max_sucursal]
print(producto_mas_vendido[['sucursal', 'id_producto', 'producto', 'cantidad']])

    sucursal id_producto                             producto  cantidad
376  Central        P399               Estuche escolar (gris)        17
488    Norte        P141     Marcador de punta fina (naranja)        17
697      Sur        P016  Cuaderno cuadriculado A5 (24 hojas)        11
923      Sur        P304    Portadocumentos con cierre (rojo)        11



Producto(s) con mayor cantidad de unidades acumuladas en cada sucursal (código, nombre y unidades vendidas), conservando todos los empates si los hubiera.

Pregunta 10 Compras por tipo de cliente. Para cada tipo de cliente, ¿cuántos clientes distintos compraron, cuántos tickets generaron y cuál fue el importe total de sus compras?

In [14]:
ventas_cli = ventas.merge(clientes, left_on='id_cliente', right_on='id', how='left')

compras_tipo_cliente = ventas_cli.groupby('tipo_cliente').agg(
    clientes_distintos=('id_cliente', 'nunique'),
    tickets_generados=('id_ticket', 'nunique'),
    importe_total=('importe', 'sum')
).reset_index()

for _, fila in compras_tipo_cliente.iterrows():
    print(
        f"{fila['tipo_cliente']}: "
        f"{fila['clientes_distintos']} clientes | "
        f"{fila['tickets_generados']} tickets | "
        f"${fila['importe_total']:.2f}"
    )

Docente: 16 clientes | 46 tickets | $899.74
Estudiante: 72 clientes | 226 tickets | $4688.00
Personal administrativo: 12 clientes | 28 tickets | $515.89



Resumen por tipo de cliente con clientes distintos, tickets generados e importe total en dólares con dos decimales.